# WSOP Main Event Analytics

## Skill, Variance, Fee Drag, and Player Segmentation

This notebook analyzes the World Series of Poker Main Event as a case study in market structure, incentive design, and performance measurement under uncertainty.

The project combines three layers:

1. **Historical tournament economics**  
   Field size, prize pool, winner share, buy-in, and estimated rake from 1971–2025.

2. **Event-time professional/amateur segmentation**  
   Separate label files classify champions and heads-up finalists as professional, amateur, or uncertain-crossover based on how they were best understood at the time of the event.

3. **Simulated skill and variance analysis**  
   A simulated player cohort tests how underlying skill can exist while realized outcomes remain noisy in small samples.

The broader goal is to treat poker as a structured analytics problem, not just a game-specific topic.

## Why This Project Matters

At surface level, this is a poker analysis. At a higher level, it is a study of market structure, incentives, and performance measurement under uncertainty.

The WSOP Main Event is useful because its economics are visible: buy-ins, prize pools, field sizes, payouts, and final placement outcomes can be tracked over time.

The project asks three questions:

1. How did the economics of the Main Event change as the tournament scaled?
2. How did payout concentration and fee drag affect participant economics?
3. How reliably does skill show up in realized outcomes once variance and sample size are considered?

These questions map beyond poker into pricing, investing, marketplace design, talent evaluation, and performance attribution.

## Questions

1. How did the WSOP Main Event change as the field expanded?
2. How did payout concentration and estimated rake change over time?
3. How often did amateur, professional, and uncertain-crossover players win or reach heads-up play?
4. How sensitive are the headline segmentation results to uncertain labels?
5. What does simulation show about skill, variance, volume, and realized outcomes?

## Data Sources and Provenance

This project began with two Kaggle WSOP Main Event datasets:

- **Full WSOP Main Event results, 1971-2025**  
  Used as the starting point for the historical economics panel.

- **Final-table results, 2001-2025**  
  Used as the starting point for heads-up finalist and late-stage player segmentation.

I used those Kaggle-derived files to create cleaner project-specific CSVs for this notebook. The processed files preserve the tournament facts needed for analysis while making the project easier to run, inspect, and reproduce.

The raw tournament data answers questions such as who finished where, in what year, for what payout, with what field size, buy-in, and prize pool. It does **not** fully answer contextual player-status questions, such as whether a player was best understood as professional, amateur, recreational, or crossover at the time of the event.

For that reason, professional/amateur segmentation is handled through a separate event-time label layer.

## File Layout

The notebook uses processed CSVs so the analysis stays readable and reproducible. Instead of hardcoding player classifications or derived tables inside Python cells, the project separates the workflow into three layers:

1. **Processed tournament tables**  
   Clean analysis-ready CSVs built from the Kaggle-derived WSOP source files.

2. **Label files**  
   Event-time professional/amateur/uncertain-crossover labels stored separately from the tournament results.

3. **Notebook analysis**  
   Reconciliation checks, summary tables, charts, simulation, and interpretation.

This separation matters because the historical results and the player-status labels answer different questions. The results data identify tournament outcomes. The label layer adds researched context. Keeping those layers separate makes the project easier to audit and revise.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / 'data' / 'processed' / 'wsop_yearly_economics_with_status.csv').exists():
            return p
    raise FileNotFoundError('Could not find project root with data/processed/wsop_yearly_economics_with_status.csv')

ROOT = find_repo_root()
DATA = ROOT / 'data'
LABELS = DATA / 'labels'
PROCESSED = DATA / 'processed'
CHARTS = ROOT / 'outputs' / 'charts'
CHARTS.mkdir(parents=True, exist_ok=True)

# Restores the original notebook's dark Plotly visual style.
pio.templates["custom_dark"] = go.layout.Template(
    layout=go.Layout(
        paper_bgcolor="#171614",
        plot_bgcolor="#1c1b19",
        font=dict(color="#cdccca", family="Arial, sans-serif"),
        colorway=["#19D3F3", "#EF553B", "#00CC96", "#7A9EAF", "#AB63FA", "#FFA15A"],
        xaxis=dict(
            gridcolor="#5c5b59",
            linecolor="#5c5b59",
            zerolinecolor="#5c5b59",
            title_font=dict(size=16),
            tickfont=dict(size=12)
        ),
        yaxis=dict(
            gridcolor="#5c5b59",
            linecolor="#5c5b59",
            zerolinecolor="#5c5b59",
            title_font=dict(size=16),
            tickfont=dict(size=12)
        ),
        margin=dict(l=70, r=40, t=120, b=80),
    )
)
pio.templates.default = "custom_dark"

era_colors = {
    "Early Era (≤1995)": "#19D3F3",
    "Pre-Boom (1996–2003)": "#7A9EAF",
    "Poker Boom (2004–2006)": "#EF553B",
    "Post-Boom (2007+)": "#2CA25F",
}
palette = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]

files = pd.DataFrame([
    {'folder':'data/processed', 'file':'wsop_yearly_economics_with_status.csv', 'role':'Year-level economics panel joined to champion status labels'},
    {'folder':'data/labels', 'file':'wsop_champion_eventtime_labels.csv', 'role':'Champion-level event-time status labels'},
    {'folder':'data/labels', 'file':'wsop_headsup_eventtime_labels.csv', 'role':'Heads-up finalist event-time status labels'},
    {'folder':'data/processed', 'file':'wsop_dataset_reconciliation_summary.csv', 'role':'Cross-source reconciliation metrics'},
    {'folder':'data/processed', 'file':'wsop_dataset_reconciliation_detail.csv', 'role':'Row-level reconciliation detail for final-table validation'},
    {'folder':'data/processed', 'file':'wsop_simulated_player_cohort.csv', 'role':'Simulated player outcomes for skill, variance, and volume analysis'},
])
files

,folder,file,role
0,data/processed,wsop_yearly_economics_with_status.csv,Year-level economics panel joined to champion status labels
1,data/labels,wsop_champion_eventtime_labels.csv,Champion-level event-time status labels
2,data/labels,wsop_headsup_eventtime_labels.csv,Heads-up finalist event-time status labels
3,data/processed,wsop_dataset_reconciliation_summary.csv,Cross-source reconciliation metrics
4,data/processed,wsop_dataset_reconciliation_detail.csv,Row-level reconciliation detail for final-table validation
5,data/processed,wsop_simulated_player_cohort.csv,"Simulated player outcomes for skill, variance, and volume analysis"


## Load Data

In [2]:
yearly = pd.read_csv(PROCESSED / 'wsop_yearly_economics_with_status.csv')
champion_labels = pd.read_csv(LABELS / 'wsop_champion_eventtime_labels.csv')
heads_up_labels = pd.read_csv(LABELS / 'wsop_headsup_eventtime_labels.csv')
sim_players = pd.read_csv(PROCESSED / 'wsop_simulated_player_cohort.csv')
reconciliation_summary = pd.read_csv(PROCESSED / 'wsop_dataset_reconciliation_summary.csv')
reconciliation_detail = pd.read_csv(PROCESSED / 'wsop_dataset_reconciliation_detail.csv')

loaded = pd.DataFrame({
    'dataset': ['yearly', 'champion_labels', 'heads_up_labels', 'sim_players', 'reconciliation_summary', 'reconciliation_detail'],
    'rows': [len(yearly), len(champion_labels), len(heads_up_labels), len(sim_players), len(reconciliation_summary), len(reconciliation_detail)],
    'columns': [yearly.shape[1], champion_labels.shape[1], heads_up_labels.shape[1], sim_players.shape[1], reconciliation_summary.shape[1], reconciliation_detail.shape[1]],
})
loaded

,dataset,rows,columns
0,yearly,54,15
1,champion_labels,55,12
2,heads_up_labels,50,11
3,sim_players,5000,7
4,reconciliation_summary,8,2
5,reconciliation_detail,225,15


## Data Validation Plan

Before interpreting any results, the notebook checks that the underlying tables loaded correctly and that the derived analysis tables are internally consistent.

The key validation questions are:

- Do the expected processed files load correctly?
- Does the yearly economics file cover the expected year range?
- Are winner payouts available in the historical panel?
- Are there duplicated year records after aggregation?
- Does the final-table source reconcile cleanly against the full-results extract where the sources overlap?
- Are final-table rows used for finalist segmentation rather than mixed into the full economics backbone?

This matters because the analysis combines multiple levels of data: event-level economics, player-level finish rows, heads-up finalist rows, and manually researched player-status labels.

## Source Reconciliation: Full Results vs. Final Table

The project uses the full-results file and final-table file for different purposes, so the notebook reconciles them before analysis.

The full-results-derived table is the backbone for long-run tournament economics because it contains the fields needed for entries, buy-in, prize pool, winner payout, and fee calculations.

The final-table-derived table is used for heads-up and late-stage player segmentation because it cleanly identifies finalists from 2001 onward.

This check confirms where the two sources overlap, where names or payouts differ after normalization, and where the final-table source contributes rows that should be used for finalist analysis rather than historical economics.

In [3]:
reconciliation_summary

In [4]:
# Rows present in the final-table source but not matched in the full-results top-9 extract.
final_table_only = reconciliation_detail.loc[reconciliation_detail['_merge'].eq('left_only')].copy()
final_table_only[['year', 'place', 'name_final_table', 'prize_final_table', 'entrants']].head(15)

### Source Decision

The yearly economics panel uses the full-results-derived source as the analytical backbone because it contains the long-run event fields needed for entries, prize pool, buy-in, winner payout, and estimated rake.

The final-table source is retained for cross-checking and for heads-up/final-table segmentation. It should not replace the full-results source for historical economics because it covers a narrower slice of the tournament and is designed around final-table outcomes rather than full event structure.

In short: **full results for economics, final-table data for finalist segmentation.**

In [5]:
# Matched rows where names or prizes differ after key normalization.
source_differences = reconciliation_detail.loc[
    reconciliation_detail['_merge'].eq('both') &
    ((~reconciliation_detail['name_match']) | (~reconciliation_detail['prize_match']))
].copy()
source_differences[['year', 'place', 'name_final_table', 'name_results', 'prize_final_table', 'prize_results', 'prize_diff']].head(20)

## Event-Time Status Labeling

The source result files contain tournament outcomes, not player occupations. They do not directly contain occupation, biography, satellite status, prior live earnings, or complete event-time career context.

For that reason, professional/amateur classification is handled through a separate label layer and joined back to tournament rows.

The labels are **event-time classifications**, not permanent lifetime identities:

- **Pro:** clearly understood as a professional poker player, online pro, or full-time poker player at the time of the event.
- **Amateur:** clearly described as amateur, recreational, or primarily tied to a non-poker occupation at the time of the event.
- **Uncertain-Crossover:** mixed evidence, usually when event-time coverage foregrounds a non-poker identity but the player later becomes clearly professional.

I used AI-assisted research to help compile event-time player context and identify relevant public descriptions of players. Those findings were not treated as raw tournament data. They were used to support a separate manually reviewed label file with confidence notes and uncertainty flags.

This separation is intentional: the raw tournament data answers who finished where and for what payout; the label layer answers a separate contextual question about how each player was best understood at the time of the event.

In [6]:
status_definitions = pd.DataFrame([
    {'status_at_event':'Pro', 'definition':'Player was meaningfully a professional poker player, professional gambler, online pro, or established tournament professional at the time of the result.'},
    {'status_at_event':'Amateur', 'definition':'Player was primarily identified by a non-poker occupation, recreational status, satellite qualification, or explicitly amateur status at the time of the result.'},
    {'status_at_event':'Uncertain-Crossover', 'definition':'Mixed evidence: serious poker ability, thin but real poker record, or later professional status while event-time identity was not cleanly pro.'},
])
status_definitions

In [7]:
label_rule_notes = pd.DataFrame([
    {'signal':'Explicit professional poker player / poker pro / online pro / professional gambler', 'default_label':'Pro', 'confidence':'High if source is clear'},
    {'signal':'Explicit amateur / recreational player', 'default_label':'Amateur', 'confidence':'High if source is clear'},
    {'signal':'Non-poker primary occupation such as accountant, logger, producer, psychologist, farmer, broker, supply-chain manager', 'default_label':'Amateur or Uncertain-Crossover', 'confidence':'Depends on prior poker résumé'},
    {'signal':'Non-poker occupation plus substantial prior Main Event or tournament evidence', 'default_label':'Uncertain-Crossover', 'confidence':'Medium unless evidence strongly supports pro'},
    {'signal':'Thin poker résumé but later professional trajectory', 'default_label':'Uncertain-Crossover', 'confidence':'Low to Medium'},
])
label_rule_notes

### Labeling Caveat

The label layer is useful, but it should not be treated as perfect ground truth.

Some players are straightforward to classify. Others are ambiguous because poker identity can change over time. A breakout Main Event result may itself be the event that moves someone from recreational or semi-professional status into a professional poker career.

For that reason, the notebook keeps confidence levels visible and treats uncertain-crossover cases separately instead of forcing every player into a false binary.

## Champion Label Table

The champion label table applies the event-time status framework to Main Event winners.

The purpose is not to argue that every champion fits neatly into a permanent identity bucket. The purpose is to understand how often the tournament was won by players who were already professionals versus players who were better understood as amateur, recreational, or crossover cases at the time of their win.

The full label file is included in `data/labels/wsop_champion_eventtime_labels.csv`. The notebook keeps the detail visible as tables instead of hardcoding the labels in Python.

In [8]:
champion_display_cols = ['year', 'player_name', 'status_at_event', 'status_basis', 'confidence', 'prior_main_event_cashes', 'entries', 'prize_usd', 'evidence_note']
champion_labels[champion_display_cols].head(12)

In [9]:
# Non-pro and crossover champion cases.
non_pro_champions = champion_labels.loc[champion_labels['status_at_event'].ne('Pro'), champion_display_cols].copy()
non_pro_champions

In [10]:
champion_status_summary = (
    champion_labels
    .groupby(['status_at_event', 'confidence'], as_index=False)
    .size()
    .rename(columns={'size':'champion_results'})
    .sort_values(['status_at_event', 'confidence'])
)
champion_status_summary

### Champion Label Interpretation

The champion results show that professional players dominate Main Event wins, but amateur and uncertain-crossover cases are still central to the tournament's history.

The strongest takeaway is not that amateurs never win. The better interpretation is that amateur breakthroughs are rare, memorable, and often tied to periods when the event's participant base or public visibility was changing.

This is why event-time labeling matters. It adds context to the raw winner list without reducing the analysis to a simplistic pro-versus-amateur story.

## Heads-Up Finalist Label Table

The heads-up finalist table includes both winners and runner-ups from 2001 through 2025.

This analysis focuses on the 2001–2025 window as a deliberate analytical choice, not a data limitation. This period captures the modern competitive era of the WSOP Main Event: defined by the online poker boom, mass participation, and data-driven strategy. Field sizes jumped from ~600 entries in 2001 to over 8,000 at peak, reflecting a structural shift in the player pool composition that makes pre- and post-2001 data fundamentally different environments. Mixing the two would combine the pre-boom live-specialist era with the internet-trained, GTO-informed modern era into a single panel.

The 2001 cutoff also enables more reliable event-time player classification. The modern press and online poker coverage ecosystem means substantially more contemporaneous documentation exists for how players were understood at the time of their result — satellite qualifications, occupation framing, prior résumés — which is central to the label methodology used here.

Pre-2001 champion labels are maintained in `data/labels/wsop_champion_eventtime_labels.csv` for context and future extension. A separate pre-2001 economics panel could be built from the full results source, but player-classification analysis for that era would require a different, more limited evidence standard.

By labeling both heads-up players from 2001 onward, the notebook can summarize whether late-stage outcomes were primarily Pro vs Pro, Pro vs Amateur, Amateur vs Pro, or other mixed-status matchups across the modern era.

In [11]:
heads_up_display_cols = ['year', 'finish', 'player_name', 'status_at_event', 'status_basis', 'confidence', 'prior_main_event_cashes', 'entries', 'prize_usd', 'evidence_note']
heads_up_labels[heads_up_display_cols].head(14)

In [12]:
heads_up_status_summary = (
    heads_up_labels
    .groupby(['finish', 'status_at_event'], as_index=False)
    .size()
    .rename(columns={'size':'results'})
    .sort_values(['finish', 'status_at_event'])
)
heads_up_status_summary

In [13]:
# Amateur or uncertain runner-up cases.
heads_up_labels.loc[
    (heads_up_labels['finish'].eq(2)) & (heads_up_labels['status_at_event'].ne('Pro')),
    heads_up_display_cols
]

### Heads-Up Interpretation

Heads-up segmentation gives a richer view than champion status alone.

If Pro vs Pro is the most common matchup type, that supports the idea that professional skill dominates late-stage outcomes. But the presence of amateur or uncertain-crossover finalists still matters because it shows that non-professional players can occasionally survive deep into the highest-leverage stage of the event.

This section is therefore less about proving a binary claim and more about describing the composition of late-stage competition.

## Special Years

Some WSOP Main Event years require special handling because they do not fit cleanly into the standard modern tournament structure.

The notebook keeps these cases visible instead of silently blending them into the analysis. This is especially important for long-run comparisons because unusual event formats can distort payout, field-size, and status interpretations.

In [14]:
special_years = pd.DataFrame([
    {'year':1970, 'treatment':'Excluded from longitudinal payout charts', 'reason':'Informal precursor event, not a standard freezeout Main Event. The champion was selected by player vote after a series of cash games. The first standard freezeout Main Event is generally recognized as 1971.'},
    {'year':2020, 'treatment':'Flagged as structural outlier', 'reason':'COVID-19 pandemic year. The event was split into an online international bracket and a live domestic bracket, with a final between the two bracket winners held in December 2020. Field size (1,379) and payout structure are not comparable to standard years.'},
])
special_years

## Yearly Economics Panel

The yearly economics panel is the backbone of the historical analysis.

Each row represents one Main Event year and contains the fields needed to analyze structural changes over time: entries, buy-in, prize pool, winner payout, winner share, estimated rake, era, and champion status.

This panel allows the notebook to evaluate how the tournament changed as participation scaled, rather than treating each year as an isolated anecdote.

In [15]:
yearly.head()